In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Cookbook_Structural_Surgery_Mutagenesis)=
# In Silico Mutagenesis

*Introducing point mutations in memory, rebuilding side chains, and evaluating structural shifts.*

Rational protein engineering frequently requires evaluating the impact of single or multiple amino acid substitutions. In silico mutagenesis requires updating topological chemical identity, generating realistic side-chain rotamers, and relaxing potential steric clashes with neighboring residues.

In this recipe, we perform an in-memory mutation on T4 Lysozyme using {func}`molsysmt.build.mutate`, rebuild missing side-chain atoms, and compare the mutant with the wild type.

:::{versionadded} 1.0.0
:::


## Loading Wild Type

We load the high-resolution T4 Lysozyme structure and inspect the target residue (Lysine 19, group index 18):

In [2]:
import molsysmt as msm

# Load wild-type T4 Lysozyme
molsys_wt = msm.convert(msm.systems['T4 lysozyme L99A']['181l.h5msm'], to_form='molsysmt.MolSys')
molsys_wt = msm.extract(molsys_wt, selection='molecule_type=="protein"')

# Inspect target residue before mutation
names, ids = msm.get(molsys_wt, element='group', selection=18, name=True, id=True)
print(f"Target residue: {names[0]} {ids[0]}")

Target residue: LYS 19


## Point Mutation

We mutate Lysine 19 to an Alanine (`ALA`) using {func}`molsysmt.build.mutate`. MolSysMT updates residue chemistry, atom topology, and removes orphaned side-chain atoms:

In [3]:
# Mutate Lys19 to Ala
molsys_mut = msm.build.mutate(molsys_wt, mutations={18: 'ALA'}, keys='group_index')

# Inspect mutated residue
mut_names, mut_ids = msm.get(molsys_mut, element='group', selection=18, name=True, id=True)
print(f"Mutated residue: {mut_names[0]} {mut_ids[0]}")
print(f"Wild-type atoms: {msm.get(molsys_wt, n_atoms=True)}, Mutant atoms: {msm.get(molsys_mut, n_atoms=True)}")

Mutated residue: ALA 19
Wild-type atoms: 1289, Mutant atoms: 1285


## Side Chain Reconstruction

We rebuild missing heavy atoms and restore hydrogen protonation states at physiological pH:

In [4]:
# Complete missing heavy atoms
molsys_mut = msm.build.add_missing_heavy_atoms(molsys_mut)

# Protonate at pH 7.4
molsys_mut = msm.build.add_missing_hydrogens(molsys_mut, pH=7.4)

msm.info(molsys_mut)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_proteins,n_structures
molsysmt.MolSys,2590,162,1,1,1,1,1,1


## Structural Superposition

We superimpose the mutant structure onto the wild type to evaluate backbone alignment and confirm structural fidelity:

In [5]:
# Align mutant onto wild-type C-alpha backbone
molsys_mut_aligned = msm.structure.least_rmsd_fit(
    molsys_mut,
    selection_fit='atom_name=="CA"',
    reference_molecular_system=molsys_wt,
    reference_selection_fit='atom_name=="CA"'
)

# Compute residual RMSD
rmsd_val = msm.structure.get_rmsd(
    molsys_mut_aligned,
    selection='atom_name=="CA"',
    reference_molecular_system=molsys_wt,
    reference_selection='atom_name=="CA"'
)
print(f"Backbone C-alpha RMSD between WT and L19A mutant: {rmsd_val[0]:.4f} nm")

Backbone C-alpha RMSD between WT and L19A mutant: 0.0000 nanometer nm


## Viewing Mutant Structure

We visualize the repaired mutant structure in 3D:

In [6]:
molsysviewer_htmlfile = '_static/views/cookbook_mutagenesis_mutant.html'


In [7]:
msm.view(molsys_mut)

'<iframe src="../../../_static/views/cookbook_mutagenesis_mutant.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

:::{seealso}
:class: dropdown

- {func}`molsysmt.build.mutate`: Performing amino acid mutations in silico.
- {func}`molsysmt.build.add_missing_heavy_atoms`: Reconstructing missing side-chain atoms.
- {func}`molsysmt.structure.least_rmsd_fit`: Superimposing mutant and wild-type structures.
:::